“The project is implemented on Databricks using Apache Spark to enable distributed processing of large-scale user–movie interaction data.”

The MovieLens dataset was obtained from the GroupLens repository and uploaded to the Databricks File System (DBFS) for distributed processing. Although the MovieLens 20M dataset contains additional files such as genome scores and user tags, this project focuses on user–movie rating interactions to implement collaborative filtering using ALS. Therefore, only the ratings and movie metadata files were used.

In [0]:
movies_df = spark.read.csv(
    "/FileStore/tables/movie.csv",
    header=True,
    inferSchema=True
)


In [0]:
ratings_df = spark.read.csv(
    "/FileStore/tables/rating.csv",
    header=True,
    inferSchema=True
)

In [0]:
ratings_df.show(5)
movies_df.show(5, truncate=False)


+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|      2|   3.5|2005-04-02 23:53:47|
|     1|     29|   3.5|2005-04-02 23:31:16|
|     1|     32|   3.5|2005-04-02 23:33:39|
|     1|     47|   3.5|2005-04-02 23:32:07|
|     1|     50|   3.5|2005-04-02 23:29:40|
+------+-------+------+-------------------+
only showing top 5 rows
+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Co

In [0]:
print("Ratings count:", ratings_df.count())
print("Movies count:", movies_df.count())


Ratings count: 20000263
Movies count: 27278


“The MovieLens 20M dataset was loaded into Apache Spark using distributed DataFrames. The ratings table contains over 20 million user–movie interactions, which necessitates distributed processing and cannot be efficiently handled using single-machine tools.”


In [0]:
ratings_df.groupBy("rating").count().orderBy("rating").show()
# To confirm ratings are on a 0.5–5 scale and check imbalance (eg. too many 4s & 5s)



+------+-------+
|rating|  count|
+------+-------+
|   0.5| 239125|
|   1.0| 680732|
|   1.5| 279252|
|   2.0|1430997|
|   2.5| 883398|
|   3.0|4291193|
|   3.5|2200156|
|   4.0|5561926|
|   4.5|1534824|
|   5.0|2898660|
+------+-------+



“Exploratory analysis was performed using Spark SQL and DataFrame operations to avoid memory bottlenecks.”

In [0]:
ratings_df.selectExpr(
    "count(distinct userId) as users",
    "count(distinct movieId) as movies"
).show()

# To see the scale of users and Sparsity challenge (key recommender insight)

+------+------+
| users|movies|
+------+------+
|138493| 26744|
+------+------+



“The ratings dataset contains over 20 million user–movie interactions across 138,493 users and 26,744 movies. The scale and sparsity of this dataset make it impractical to process using traditional single-machine tools, thereby motivating the use of Apache Spark for distributed data processing and machine learning.”

Possible Interactions 138493*26744 = 3.7 billion user-movie interactions, only 20 million ratings exist. 

In [0]:
ratings_df.groupBy("userId").count().describe().show()


+-------+-----------------+------------------+
|summary|           userId|             count|
+-------+-----------------+------------------+
|  count|           138493|            138493|
|   mean|          69247.0| 144.4135299257002|
| stddev|39979.62975274626|230.26725699673477|
|    min|                1|                20|
|    max|           138493|              9254|
+-------+-----------------+------------------+



- Average ratings per user  = 144  mean
- Minimum ratings per user = 20
- Maximum ratings per user = 9,254
- Std deviation ≈ 230
This suggests highly uneven user activity as some users rate only 20 movies while some rate as high as 9254 movies. Small number of power users who rate the movies and very large number of casual users who do not rate. 
Even active users interact with only a tiny fraction of the 26,744 movies. 

“Analysis of user activity revealed a highly skewed distribution of ratings per user. While the average user rated approximately 144 movies, some users rated over 9,000 movies, whereas the minimum was only 20 ratings per user. This uneven interaction pattern further contributes to matrix sparsity and supports the use of matrix factorization techniques such as ALS.”

In [0]:
#popular movies 
ratings_df.groupBy("movieId").count() \
    .orderBy("count", ascending=False) \
    .show(10)


+-------+-----+
|movieId|count|
+-------+-----+
|    296|67310|
|    356|66172|
|    318|63366|
|    593|63299|
|    480|59715|
|    260|54502|
|    110|53769|
|    589|52244|
|   2571|51334|
|    527|50054|
+-------+-----+
only showing top 10 rows


Data Cleaning 

In [0]:
# drop nulls
ratings_clean = ratings_df.dropna()
movies_clean = movies_df.dropna()



In [0]:
ratings_clean = ratings_clean.select("userId", "movieId", "rating")
movies_clean = movies_clean.select("movieId", "title", "genres")


In [0]:

# cache clean ratings 
ratings_clean.cache()
ratings_clean.count()  # triggers caching

20000263

In [0]:
%skip
final_df = ratings_clean.join(movies_clean, on="movieId")

“The dataset was split into training and testing sets using Spark’s distributed random split function.”

In [0]:
train_df, test_df = ratings_clean.randomSplit([0.8, 0.2], seed=42)

print("Train count:", train_df.count())
print("Test count:", test_df.count())


Train count: 15998512
Test count: 4001751


In [0]:
from pyspark.ml.recommendation import ALS


In [0]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=10,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop",
    nonnegative=True
)


- userId → who is watching
- movieId → what is being watched
- rating → how much they liked it



Iter = 10 “Repeat the back-and-forth learning 10 times.”

regParam=0.1, This prevents the model from: Overreacting to noisy ratings and Memorizing weird users

 coldStartStrategy="drop",

 Making predictions for users or movies the model has never seen

Producing NaN errors

 nonnegative=True No weird negative preference scores





In [0]:
als_model = als.fit(train_df)


“Model performance was evaluated using Root Mean Squared Error (RMSE).”

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = als_model.transform(test_df)

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print(f"RMSE: {rmse}")


RMSE: 0.8159007999235753


In [0]:
user_recs = als_model.recommendForAllUsers(5)
user_recs.show(10, truncate=False)


+------+--------------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                         |
+------+--------------------------------------------------------------------------------------------------------+
|1     |[{126219, 5.6064773}, {129536, 5.3485894}, {95776, 5.2810073}, {73529, 5.194863}, {95595, 5.148286}]    |
|6     |[{126219, 7.104436}, {101855, 5.3167377}, {113041, 5.2394905}, {95776, 5.1578336}, {60356, 5.1155486}]  |
|12    |[{126219, 6.4440084}, {95776, 5.1499467}, {129536, 4.9314895}, {101538, 4.9226613}, {101855, 4.8174744}]|
|13    |[{126219, 7.031175}, {129536, 5.6239247}, {95776, 5.4940925}, {117873, 5.1491923}, {60356, 5.142237}]   |
|16    |[{126219, 6.383647}, {95776, 5.028787}, {101855, 4.950442}, {101538, 4.939309}, {109887, 4.8873215}]    |
|22    |[{126219, 6.2377663}, {129536, 5.5488124}, {95776, 5.236154}, {95595, 5.0348735}

In [0]:
from pyspark.sql.functions import explode

user_recs_exploded = user_recs \
    .select("userId", explode("recommendations").alias("rec")) \
    .select("userId", "rec.movieId", "rec.rating")

final_recs = user_recs_exploded.join(
    movies_clean,
    on="movieId",
    how="left"
)

final_recs.show(10, truncate=False)


+-------+------+---------+----------------------------------------+-----------------------+
|movieId|userId|rating   |title                                   |genres                 |
+-------+------+---------+----------------------------------------+-----------------------+
|126219 |1     |5.6064773|Marihuana (1936)                        |Documentary|Drama      |
|129536 |1     |5.3485894|Code Name Coq Rouge (1989)              |(no genres listed)     |
|95776  |1     |5.2810073|Bob Funk (2009)                         |Comedy|Romance         |
|73529  |1     |5.194863 |God’s Wedding (As Bodas de Deus) (1999) |Comedy                 |
|95595  |1     |5.148286 |Bela Kiss: Prologue (2013)              |Horror|Mystery|Thriller|
|126219 |6     |7.104436 |Marihuana (1936)                        |Documentary|Drama      |
|101855 |6     |5.3167377|Shepard & Dark (2012)                   |Documentary            |
|113041 |6     |5.2394905|Father and Guns (De père en flic) (2009)|Comedy|Crime 

“This project demonstrates an end-to-end big data recommendation pipeline using Apache Spark, highlighting the need for distributed processing when working with large-scale user interaction data.”

“An end-to-end Spark-based collaborative filtering pipeline was implemented using ALS on the MovieLens 20M dataset, generating personalized top-N movie recommendations at scale.”


“I used Spark’s ALS algorithm to factorize a large, sparse user–movie rating matrix into latent user and movie features. The model was trained on 20 million ratings and evaluated using RMSE. Finally, I generated top-N personalized recommendations by predicting unseen user–movie interactions.”